[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsinghlab/pyaging/blob/main/tutorials/tutorial_cpgptgrimage3.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/rsinghlab/pyaging/blob/main/tutorials/tutorial_cpgptgrimage3.ipynb)

# 🧬 ⚙️ CpGPT Quick Setup Tutorial ⚙️ 🧬

Welcome to the CpGPT Quick Setup Tutorial! 👋 

In this notebook, we'll walk you through the fastest way of using CpGPT for your research.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Retrieve DNA LLM Embeddings](#2-retrieve-dna-llm-embeddings)
3. [Download and Load Model](#3-download-and-load-model)
4. [Prepare Data Objects](#4-prepare-data-objects)
5. [Run Inference](#5-run-inference)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements.

In [55]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
DEPENDENCIES_DIR = "../dependencies"
LLM_DEPENDENCIES_DIR = DEPENDENCIES_DIR + "/human"
DATA_DIR = "../data"
PROCESSED_DIR = "../data/tutorials/processed/quick_setup"

MODEL_NAME = "cancer"
MODEL_CHECKPOINT_PATH = f"../dependencies/model/weights/{MODEL_NAME}.ckpt"
MODEL_CONFIG_PATH = f"../dependencies/model/config/{MODEL_NAME}.yaml"
MODEL_VOCAB_PATH = f"../dependencies/model/vocab/{MODEL_NAME}.json"

ARROW_DF_PATH = "../data/cpgcorpus/raw/GSE182215/GPL13534/betas/QCDPB.arrow"
ARROW_DF_FILTERED_PATH = "../data/tutorials/raw/toy_filtered.arrow"

# The maximum context length to give to the model
MAX_INPUT_LENGTH = 20_000 # you might wanna go higher hardware permitting
MAX_ATTN_LENGTH = 1_000

> **⚠️ Warning**
> 
> It is recommended to have a GPU for inference as CPU might be slow.
> 
> Reconstructing the methylome for a few hundred samples might take up to one hour on a CPU. ⌛
>
> This might be a great exercise in testing your patience.

### 1.2 Import packages


In [56]:
# Standard library imports
import warnings
import os
import json

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyaging as pya
import seaborn as sns
import torch

# Lightning imports
from lightning.pytorch import seed_everything

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.data.cpgpt_datamodule import CpGPTDataModule
from cpgpt.trainer.cpgpt_trainer import CpGPTTrainer
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.model.cpgpt_module import m_to_beta

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)
try:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except:
    pass

Seed set to 42


## 2. Retrieve DNA LLM Embeddings

To retrieve the DNA LLM Embeddings, there are two options:
- download the dependencies with all of the sequence embeddings for the CpG sites targeted by the Illumina arrays;
- generate from scratch using the DNA LLM directly for loci outside of the ones already available for download.

In [3]:
# First let's declare the inferencer
inferencer = CpGPTInferencer(dependencies_dir=DEPENDENCIES_DIR, data_dir=DATA_DIR)

cpgpt -CpGPTInferencer: There are 2089 GSE datasets available such as GSE100184, GSE100208, GSE100209, etc.


### 2.1 Download Dependencies


The already-processed dependencies contain the sequence embeddings for both human (`s3://cpgpt-lucascamillo-public/dependencies/human`) and several mammalian species (`s3://cpgpt-lucascamillo-public/dependencies/mammalian`). Here, let's use the human as an example:

In [4]:
inferencer.download_dependencies(species="human")

cpgpt -CpGPTInferencer: All 3 dependency files for human already exist at ../dependencies/human. Skipping download.


### 2.1.1 Inspect Dependencies Metadata Files

Let's examine the contents of `ensembl_metadata.db` and `illumina_metadata.db` to understand what metadata is stored.

**How are these files generated?**
- **`ensembl_metadata.db`**: Created by `DNALLMEmbedder._parse_ensembl_metadata()` which parses downloaded genome assembly info from Ensembl (chromosome names, sequences, etc.)
- **`illumina_metadata.db`**: Created by `IlluminaMethylationProber.parse_illumina_metadata()` which downloads Illumina array manifests and maps probe IDs to genomic locations
- Both are **downloaded from AWS S3** via `inferencer.download_dependencies()` (pre-computed for convenience)

In [33]:
# Inspect ensembl_metadata.db and illumina_metadata.db
import sqlitedict
from pathlib import Path

dependencies_dir = Path(LLM_DEPENDENCIES_DIR)

# 1. Inspect ensembl_metadata.db
ensembl_file = dependencies_dir / "ensembl_metadata.db"
print(f"File: {ensembl_file}")
print(f"File size: {ensembl_file.stat().st_size / (1024**2):.2f} MB")

File: ../dependencies/human/ensembl_metadata.db
File size: 887.38 MB


In [39]:
with sqlitedict.SqliteDict(ensembl_file, autocommit=False) as db:
    ensembl_metadata = dict(db)
    print(f"\nTotal species: {len(ensembl_metadata)}")
    print(f"\nFirst 10 species keys:")
    for i, key in enumerate(list(ensembl_metadata.keys())[:10]):
        print(f"  {i+1}. {key}")


Total species: 324

First 10 species keys:
  1. acanthochromis_polyacanthus
  2. accipiter_nisus
  3. ailuropoda_melanoleuca
  4. amazona_collaria
  5. amphilophus_citrinellus
  6. amphiprion_ocellaris
  7. amphiprion_percula
  8. anabas_testudineus
  9. anas_platyrhynchos
  10. anas_platyrhynchos_platyrhynchos


In [21]:
ensembl_metadata.keys()

dict_keys(['acanthochromis_polyacanthus', 'accipiter_nisus', 'ailuropoda_melanoleuca', 'amazona_collaria', 'amphilophus_citrinellus', 'amphiprion_ocellaris', 'amphiprion_percula', 'anabas_testudineus', 'anas_platyrhynchos', 'anas_platyrhynchos_platyrhynchos', 'anas_zonorhyncha', 'anolis_carolinensis', 'anser_brachyrhynchus', 'anser_cygnoides', 'aotus_nancymaae', 'apteryx_haastii', 'apteryx_owenii', 'apteryx_rowi', 'aquila_chrysaetos_chrysaetos', 'astatotilapia_calliptera', 'astyanax_mexicanus', 'astyanax_mexicanus_pachon', 'athene_cunicularia', 'balaenoptera_musculus', 'betta_splendens', 'bison_bison_bison', 'bos_grunniens', 'bos_indicus_hybrid', 'bos_mutus', 'bos_taurus', 'bos_taurus_hybrid', 'bubo_bubo', 'buteo_japonicus', 'caenorhabditis_elegans', 'cairina_moschata_domestica', 'calidris_pugnax', 'calidris_pygmaea', 'callithrix_jacchus', 'callorhinchus_milii', 'camarhynchus_parvulus', 'camelus_dromedarius', 'canis_lupus_dingo', 'canis_lupus_familiaris', 'canis_lupus_familiarisbasenji

In [42]:
ensembl_metadata['homo_sapiens'].keys()

dict_keys(['scientific_name', 'assembly', 'vocab', 'reverse_vocab', 'nucleotide-transformer-v2-500m-multi-species', 'DNABERT-2-117M', 'hyenadna-large-1m-seqlen-hf', 'evo-1-8k-base'])

In [43]:
# vocab: maps chromosome/sequence names → integer indices.
# Why 136,214? The human genome (GRCh38) contains much more than 24 chromosomes
ensembl_metadata['homo_sapiens']

{'scientific_name': 'Homo sapiens',
 'assembly': 'GRCh38',
 'vocab': {'1': 0,
  '10': 1,
  '11': 2,
  '12': 3,
  '13': 4,
  '14': 5,
  '15': 6,
  '16': 7,
  '17': 8,
  '18': 9,
  '19': 10,
  '2': 11,
  '20': 12,
  '21': 13,
  '22': 14,
  '3': 15,
  '4': 16,
  '5': 17,
  '6': 18,
  '7': 19,
  '8': 20,
  '9': 21,
  'AADB01066164.1.1.35954': 22,
  'AADC01095577.1': 23,
  'AADD01112371.1': 24,
  'AADD01115518.1': 25,
  'AADD01116787.1': 26,
  'AADD01116788.1': 27,
  'AADD01116830.1.1.32292': 28,
  'AADD01117999.1': 29,
  'AADD01118406.1': 30,
  'AADD01118410.1': 31,
  'AADD01172788.1': 32,
  'AADD01172789.1': 33,
  'AADD01172902.1': 34,
  'AADD01209098.1': 35,
  'AB000878.1.1.33983': 36,
  'AB000879.1.1.39436': 37,
  'AB000880.1.1.22973': 38,
  'AB001517.1': 39,
  'AB001523.1': 40,
  'AB014077.1.1.30452': 41,
  'AB014078.1.1.22738': 42,
  'AB014079.1.1.13976': 43,
  'AB014080.1.1.32114': 44,
  'AB014081.1.1.13829': 45,
  'AB014082.1.1.29591': 46,
  'AB014083.1.1.22715': 47,
  'AB014084.1.1

In [46]:
# Maps genomic location → row index in 2001bp_dna_embeddings.mmap
ensembl_metadata['homo_sapiens']['nucleotide-transformer-v2-500m-multi-species']

{2001: {'16:53434199': 0,
  '3:172198246': 1,
  '7:2550930': 2,
  '9:92248272': 3,
  '1:90729116': 4,
  '17:56456886': 5,
  '8:42405775': 6,
  '14:68874421': 7,
  '16:28878778': 8,
  '8:41310282': 9,
  '1:230425046': 10,
  '1:28136152': 11,
  '9:98701950': 12,
  '8:132086254': 13,
  '15:22838619': 14,
  '9:137103471': 15,
  '19:54191826': 16,
  '6:25282550': 17,
  '1:230268566': 18,
  '12:123601929': 19,
  '4:155467052': 20,
  '4:153688704': 21,
  '15:59493106': 22,
  '8:48978049': 23,
  '1:5877192': 24,
  '20:6769958': 25,
  '8:86069323': 26,
  'X:23065268': 27,
  '3:81684888': 28,
  '11:70700049': 29,
  '8:107101526': 30,
  '7:87623207': 31,
  '3:15065202': 32,
  '14:59923067': 33,
  '16:3413963': 34,
  '20:50342466': 35,
  '1:166989201': 36,
  '1:213997032': 37,
  '1:43365369': 38,
  '20:17399503': 39,
  '1:25052407': 40,
  '18:63383353': 41,
  '19:54223320': 42,
  '14:37197283': 43,
  '1:50569192': 44,
  '10:79068944': 45,
  '1:200042657': 46,
  '2:11344578': 47,
  '12:30691297': 4

In [44]:
# Look at homo_sapiens metadata in detail
human_metadata = ensembl_metadata.get("homo_sapiens", {})

print(f"Scientific name: {human_metadata.get('scientific_name')}")
print(f"Assembly: {human_metadata.get('assembly')}")

print(f"\nChromosome vocabulary (name → index):")
vocab = human_metadata.get("vocab", {})
print(f"  Total chromosomes: {len(vocab)}")
for chrom, idx in list(vocab.items())[:10]:
    print(f"  '{chrom}' → {idx}")
print("  ...")

print(f"\nReverse vocabulary (index → name):")
reverse_vocab = human_metadata.get("reverse_vocab", {})
for idx, chrom in list(reverse_vocab.items())[:10]:
    print(f"  {idx} → '{chrom}'")
print("  ...")

print(f"\nDNA LLM entries:")
for llm in ["nucleotide-transformer-v2-500m-multi-species", "DNABERT-2-117M", "hyenadna-large-1m-seqlen-hf"]:
    if llm in human_metadata:
        llm_data = human_metadata[llm]
        print(f"  {llm}:")
        print(f"    Context lengths available: {list(llm_data.keys())}")
        if llm_data:
            first_ctx = list(llm_data.keys())[0]
            print(f"    Embeddings for {first_ctx}bp: {len(llm_data[first_ctx])} genomic locations")

Scientific name: Homo sapiens
Assembly: GRCh38

Chromosome vocabulary (name → index):
  Total chromosomes: 136214
  '1' → 0
  '10' → 1
  '11' → 2
  '12' → 3
  '13' → 4
  '14' → 5
  '15' → 6
  '16' → 7
  '17' → 8
  '18' → 9
  ...

Reverse vocabulary (index → name):
  0 → '1'
  1 → '10'
  2 → '11'
  3 → '12'
  4 → '13'
  5 → '14'
  6 → '15'
  7 → '16'
  8 → '17'
  9 → '18'
  ...

DNA LLM entries:
  nucleotide-transformer-v2-500m-multi-species:
    Context lengths available: [2001]
    Embeddings for 2001bp: 1185347 genomic locations
  DNABERT-2-117M:
    Context lengths available: []
  hyenadna-large-1m-seqlen-hf:
    Context lengths available: []


In [30]:
# 2. Inspect illumina_metadata.db
illumina_file = dependencies_dir / "illumina_metadata.db"
print(f"File: {illumina_file}")
print(f"File size: {illumina_file.stat().st_size / (1024**2):.2f} MB")

ILLUMINA METADATA (illumina_metadata.db)
File: ../dependencies/human/illumina_metadata.db
File size: 30.26 MB


In [31]:
with sqlitedict.SqliteDict(illumina_file, autocommit=False) as db:
    illumina_metadata = dict(db)
    print(f"\nTop-level keys: {list(illumina_metadata.keys())}")


Top-level keys: ['processed_files', 'homo_sapiens']


In [32]:
# Look at homo_sapiens Illumina metadata in detail
human_illumina = illumina_metadata.get("homo_sapiens", {})
print("=" * 70)
print("HOMO SAPIENS ILLUMINA PROBE MAPPING")
print("=" * 70)
print(f"Total probes mapped: {len(human_illumina)}")

print(f"\nFirst 10 probe mappings (probe_id → genomic_location):")
for i, (probe_id, location) in enumerate(list(human_illumina.items())[:10]):
    print(f"  {probe_id} → {location}")

print(f"\nExample probe lookup:")
example_probe = list(human_illumina.keys())[0]
print(f"  Probe ID: {example_probe}")
print(f"  Genomic location: {human_illumina[example_probe]}")

HOMO SAPIENS ILLUMINA PROBE MAPPING
Total probes mapped: 1186366

First 10 probe mappings (probe_id → genomic_location):
  cg00000029 → 16:53434199
  cg00000109 → 3:172198246
  cg00000155 → 7:2550930
  cg00000158 → 9:92248272
  cg00000165 → 1:90729116
  cg00000221 → 17:56456886
  cg00000236 → 8:42405775
  cg00000289 → 14:68874421
  cg00000292 → 16:28878778
  cg00000321 → 8:41310282

Example probe lookup:
  Probe ID: cg00000029
  Genomic location: 16:53434199


### 2.2 Generate DNA LLM Embeddings


To generate genomic embeddings for loci outside of the ones already available for download, we can use the `DNALLMEmbedder` class. We need the loci in a list with the following format from ENSEMBL: 'chromosome:position'. Be mindful as this function can take a long time to run dependending on your GPU. For instance, embeddings ~1M genomic loci from the Illumina arrays takes about 12h in an RTX 4090.

In [5]:
if not os.path.exists(LLM_DEPENDENCIES_DIR):

    # List CpG genomic locations
    example_genomic_locations = ['1:100000', '1:250500', 'X:2031253']

    # Declare required class
    embedder = DNALLMEmbedder(dependencies_dir=LLM_DEPENDENCIES_DIR)

    # Parse the embeddings
    embedder.parse_dna_embeddings(
        example_genomic_locations,
        "homo_sapiens",
        dna_llm="nucleotide-transformer-v2-500m-multi-species",
        dna_context_len=2001,
    )

In [6]:
os.path.exists(LLM_DEPENDENCIES_DIR)

True

### 2.3 Inspect DNA Embeddings File

Let's examine the pre-computed DNA embeddings stored in the `.mmap` file.

In [7]:
# Inspect DNA embeddings .mmap file
import numpy as np
from pathlib import Path

# Define paths
species = "homo_sapiens"
dna_llm = "nucleotide-transformer-v2-500m-multi-species"
dna_context_len = 2001
embedding_size = 1024  # Nucleotide Transformer v2 500M embedding dimension

embeddings_dir = Path(LLM_DEPENDENCIES_DIR) / "dna_embeddings" / species / dna_llm
embeddings_file = embeddings_dir / f"{dna_context_len}bp_dna_embeddings.mmap"

print(f"Embeddings file: {embeddings_file}")
print(f"File exists: {embeddings_file.exists()}")
print(f"File size: {embeddings_file.stat().st_size / (1024**3):.2f} GB")

Embeddings file: ../dependencies/human/dna_embeddings/homo_sapiens/nucleotide-transformer-v2-500m-multi-species/2001bp_dna_embeddings.mmap
File exists: True
File size: 4.88 GB


In [12]:
# Load the embeddings index dictionary to get the number of genomic locations
# Create a DNALLMEmbedder instance to access the metadata
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder

embedder_inspect = DNALLMEmbedder(dependencies_dir=LLM_DEPENDENCIES_DIR)
embeddings_index_dict = embedder_inspect.ensembl_metadata_dict[species][dna_llm][dna_context_len]

cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


In [13]:
print("=" * 60)
print("EMBEDDINGS INDEX DICTIONARY")
print("=" * 60)
print(f"Total genomic locations: {len(embeddings_index_dict)}")
print(f"Type: {type(embeddings_index_dict)}")

# Show first 10 entries
print(f"\nFirst 10 entries (genomic_location → row_index):")
for i, (loc, idx) in enumerate(list(embeddings_index_dict.items())[:10]):
    print(f"  '{loc}' → {idx}")

EMBEDDINGS INDEX DICTIONARY
Total genomic locations: 1185347
Type: <class 'dict'>

First 10 entries (genomic_location → row_index):
  '16:53434199' → 0
  '3:172198246' → 1
  '7:2550930' → 2
  '9:92248272' → 3
  '1:90729116' → 4
  '17:56456886' → 5
  '8:42405775' → 6
  '14:68874421' → 7
  '16:28878778' → 8
  '8:41310282' → 9


In [17]:
# Memory-map the DNA embeddings file
n_locations = len(embeddings_index_dict)
dna_embeddings = np.memmap(
    embeddings_file,
    dtype="float32",
    mode="r",
    shape=(n_locations, embedding_size)
)

print("=" * 60)
print("DNA EMBEDDINGS (.mmap)")
print("=" * 60)
print(f"Shape: {dna_embeddings.shape}")
print(f"  - {dna_embeddings.shape[0]} genomic locations (CpG sites)")
print(f"  - {dna_embeddings.shape[1]} embedding dimensions")
print(f"Dtype: {dna_embeddings.dtype}")
print(f"Memory-mapped: Data stays on disk until accessed")

DNA EMBEDDINGS (.mmap)
Shape: (1185347, 1024)
  - 1185347 genomic locations (CpG sites)
  - 1024 embedding dimensions
Dtype: float32
Memory-mapped: Data stays on disk until accessed


In [16]:
dna_embeddings

memmap([[-0.07362102,  0.05253434, -0.05064404, ..., -0.06335837,
          0.32719395,  0.06025311],
        [-0.11024167,  0.03513953, -0.12368321, ..., -0.11169019,
          0.23786864,  0.02212111],
        [-0.02916887,  0.07912867, -0.13034678, ..., -0.25844496,
          0.25982502,  0.04692779],
        ...,
        [-0.12993048,  0.03785931, -0.0960447 , ..., -0.11420815,
          0.20444076,  0.05152277],
        [-0.02355557,  0.0636318 , -0.1008666 , ..., -0.2387739 ,
          0.24411343,  0.02425192],
        [-0.05697441,  0.0392761 , -0.19313236, ..., -0.07861166,
          0.23701754,  0.05164669]], shape=(1185347, 1024), dtype=float32)

In [15]:
# Look up a specific genomic location
example_location = list(embeddings_index_dict.keys())[0]
example_index = embeddings_index_dict[example_location]

print("=" * 60)
print("EXAMPLE: Look Up Embedding by Genomic Location")
print("=" * 60)
print(f"Genomic location: '{example_location}'")
print(f"Row index in mmap: {example_index}")
print(f"Embedding vector shape: {dna_embeddings[example_index].shape}")
print(f"Embedding vector (first 10 values): {dna_embeddings[example_index, :10]}")

EXAMPLE: Look Up Embedding by Genomic Location
Genomic location: '16:53434199'
Row index in mmap: 0
Embedding vector shape: (1024,)
Embedding vector (first 10 values): [-0.07362102  0.05253434 -0.05064404 -0.01835526 -0.1005891   0.11271825
 -0.12652288 -0.6178616   0.06389362 -0.05463275]


## 3. Download and Load Model

Please first check the model zoo for the available models and their corresponding features on the README.md file. To load any given model, you first need to define the dictionary structure with the hyperparameters and use the `CpGPTInferencer` class.

### 3.1 Download Checkpoint and Configuration Files

In [55]:
# Download the checkpoint and configuration files
inferencer.download_model(MODEL_NAME)

cpgpt -CpGPTInferencer: Successfully downloaded model 'cancer'.


In [56]:
inferencer.download_model('small')

cpgpt -CpGPTInferencer: Successfully downloaded model 'small'.


### 3.2 Load Model

In [57]:
# Load the model configuration
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)

# Load the model weights
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

cpgpt -CpGPTInferencer: Checkpoint loaded into the model.


## 4 Prepare Data Objects

In order to perform inference, we need to prepare the data objects, which are essentially memory-mapped versions for faster loading. As an example, let's download a toy dataset from the _CpGCorpus_ database.

### 4.1 Download and Load Toy Data

In [58]:
inferencer.download_cpgcorpus_dataset("GSE182215")

cpgpt -CpGPTInferencer: All 2 files for dataset GSE182215 already exist at ../data/cpgcorpus/raw/GSE182215. Skipping download.


There is no need to impute the methylation data for CpGPT -- it simply ignores the missing values.

In [59]:
df = pd.read_feather(ARROW_DF_PATH)
df.set_index('GSM_ID', inplace=True)
df.head()

,cg00000029,cg00000108,cg00000109,cg00000165,cg00000236,cg00000289,cg00000292,cg00000321,cg00000363,cg00000622,...,rs7746156,rs798149,rs845016,rs877309,rs9292570,rs9363764,rs939290,rs951295,rs966367,rs9839873
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM5525203,0.324878,0.958127,0.853575,NaN,0.854126,NaN,0.791321,0.249139,0.291044,0.013388,...,0.025733,0.462424,0.100629,0.979795,0.019506,0.947983,0.054746,0.495169,0.675477,0.782523
GSM5525204,0.298781,0.939107,0.897159,0.217733,0.874908,NaN,0.828892,0.228412,0.397047,0.013511,...,0.978182,0.017012,0.930166,0.568672,0.493534,0.042713,0.620214,0.511735,0.069878,0.941439
GSM5525205,NaN,NaN,NaN,NaN,0.472385,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM5525206,0.125208,0.961126,0.822223,0.229362,0.861563,NaN,0.877324,0.178019,0.377428,0.017229,...,0.550734,0.444213,0.644010,0.563410,0.460727,0.961204,0.051867,0.487339,0.126543,0.899515
GSM5525207,0.278861,0.970059,0.929905,0.171255,0.907603,0.820531,0.893471,0.185116,0.377647,0.012591,...,0.495878,0.020168,0.483792,0.021345,0.015444,0.968064,0.551504,0.970979,0.062037,0.764114


In [61]:
df.shape

(38, 485578)

### 4.2 Filter Vocab Features and Save Data

While not strictly required, filtering for the features used in finetuning gives you the best chance of achieving good performance.

In [62]:
# Load list
vocab = json.load(open(MODEL_VOCAB_PATH, 'r'))

In [63]:
vocab['input']

['cg00000292',
 'cg00002426',
 'cg00003994',
 'cg00005847',
 'cg00008493',
 'cg00009407',
 'cg00011459',
 'cg00012199',
 'cg00012386',
 'cg00012792',
 'cg00013618',
 'cg00014085',
 'cg00014837',
 'cg00015770',
 'cg00019495',
 'cg00020533',
 'cg00021527',
 'cg00022866',
 'cg00024396',
 'cg00024812',
 'cg00025991',
 'cg00027083',
 'cg00027674',
 'cg00029826',
 'cg00031162',
 'cg00032227',
 'cg00033516',
 'cg00033773',
 'cg00034039',
 'cg00035347',
 'cg00035623',
 'cg00037763',
 'cg00037940',
 'cg00040861',
 'cg00040873',
 'cg00043004',
 'cg00043080',
 'cg00044245',
 'cg00047050',
 'cg00050312',
 'cg00051979',
 'cg00054706',
 'cg00056767',
 'cg00057593',
 'cg00058938',
 'cg00059424',
 'cg00059930',
 'cg00060762',
 'cg00061059',
 'cg00062776',
 'cg00063144',
 'cg00065385',
 'cg00065408',
 'cg00066816',
 'cg00067471',
 'cg00069261',
 'cg00071250',
 'cg00072216',
 'cg00075967',
 'cg00076645',
 'cg00077877',
 'cg00078194',
 'cg00079056',
 'cg00079563',
 'cg00080012',
 'cg00081935',
 'cg000846

In [64]:
df = df.loc[:, df.columns.isin(vocab['input'])]
df.head()

,cg00000292,cg00002426,cg00003994,cg00005847,cg00008493,cg00009407,cg00011459,cg00012199,cg00012386,cg00012792,...,cg27650175,cg27650434,cg27652350,cg27653134,cg27654142,cg27655905,cg27657283,cg27662379,cg27662877,cg27665659
GSM_ID,,,,,,,,,,,,,,,,,,,,,
GSM5525203,0.791321,0.905377,0.091164,0.090651,0.951774,0.048334,0.938640,0.035517,0.056877,0.087570,...,0.045768,0.072923,0.132974,0.949820,0.065028,0.063921,0.052416,0.077480,0.043270,0.057590
GSM5525204,0.828892,0.964620,0.042569,0.125086,0.945982,0.052446,0.951808,0.052040,0.063641,0.102222,...,0.053291,0.089544,0.130468,0.959386,0.066889,0.055794,0.044713,0.069569,0.039361,0.070515
GSM5525205,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.058280,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GSM5525206,0.877324,0.920315,0.042593,0.201401,0.951397,0.058045,0.947452,0.051123,0.060039,0.091149,...,0.046296,0.112071,0.096274,0.927876,0.088435,0.056817,0.050824,0.068512,0.070187,0.063018
GSM5525207,0.893471,0.943485,0.039004,0.139082,0.952827,0.045738,0.950823,0.036847,0.041898,0.082241,...,0.047588,0.105776,0.114468,0.940735,0.057654,0.045630,0.036157,0.051430,0.041078,0.082783


In [65]:
df.shape

(38, 19948)

In [66]:
df.to_feather(ARROW_DF_FILTERED_PATH)

### 4.3 Memory-Map Data

In order to perform inference, we need to memory-map the data. This is done by using the `CpGPTDataSaver` class. We first need to define the `DNALLMEmbedder` and `IlluminaMethylationProber` classes, which contain the information about the DNA LLM Embeddings and the conversion between Illumina array probes to genomic locations, respectively.

In [20]:
embedder = DNALLMEmbedder(dependencies_dir=LLM_DEPENDENCIES_DIR)

cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


In [21]:
prober = IlluminaMethylationProber(dependencies_dir=LLM_DEPENDENCIES_DIR, embedder=embedder)

cpgpt -IlluminaMethylationProber: Illumina metadata dictionary loaded successfully.


In [22]:
# Define datasaver
quick_setup_datasaver = CpGPTDataSaver(data_paths=ARROW_DF_FILTERED_PATH, processed_dir=PROCESSED_DIR)

# Process the file (creates dataset_metrics.db, genomic_locations.db, and dataset folder)
quick_setup_datasaver.process_files(prober, embedder)

cpgpt -CpGPTDataSaver: File processing completed.


### 4.3.1 Inspect Processed Data

Let's examine the contents of the generated database files to understand what was created.

In [58]:
# Inspect dataset_metrics.db and genomic_locations.db
import sqlitedict
from pathlib import Path

processed_dir = Path(PROCESSED_DIR)

# 1. Inspect dataset_metrics.db
with sqlitedict.SqliteDict(processed_dir / "dataset_metrics.db", autocommit=False) as db:
    dataset_metrics = dict(db)
    print(f"Number of datasets: {len(db)}\n")
    for data_path, metrics in db.items():
        print(f"Dataset: {data_path}")
        print("-" * 40)
        for key, value in metrics.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.6f}")
            else:
                print(f"  {key}: {value}")

Number of datasets: 1

Dataset: ../data/tutorials/raw/toy_filtered.arrow
----------------------------------------
  species: homo_sapiens
  num_samples: 38
  X_mean: 0.285787
  X_max: 0.997802
  X_median: 0.083224
  X_min: 0.002200
  X_std: 0.341532
  original_number_of_features: 19948
  filtered_number_of_features: 19948


In [59]:
dataset_metrics

{'../data/tutorials/raw/toy_filtered.arrow': {'species': 'homo_sapiens',
  'num_samples': 38,
  'X_mean': 0.2857868357093941,
  'X_max': 0.9978016626738487,
  'X_median': 0.08322424260783227,
  'X_min': 0.0021999456357507325,
  'X_std': 0.34153211487292384,
  'original_number_of_features': 19948,
  'filtered_number_of_features': 19948}}

In [60]:
# 2. Inspect genomic_locations.db
with sqlitedict.SqliteDict(processed_dir / "genomic_locations.db", autocommit=False) as db:
    species_locations = dict(db)
    for species, locations in db.items():
        print(f"Species: {species}")
        print(f"Total genomic locations: {len(locations)}")
        print(f"First 5 locations: {locations[:5]}")
        print(f"Last 5 locations: {locations[-5:]}")

Species: homo_sapiens
Total genomic locations: 19948
First 5 locations: ['14:85529756', '1:43707906', '14:105169431', '16:4848304', '17:29588953']
Last 5 locations: ['20:1994277', '6:73654425', '8:124474813', '13:51222086', '18:59900038']


In [61]:
species_locations

{'homo_sapiens': ['14:85529756',
  '1:43707906',
  '14:105169431',
  '16:4848304',
  '17:29588953',
  '10:17129695',
  '17:49678408',
  '20:62805456',
  '11:64233621',
  '8:69466504',
  '16:82170785',
  '4:170090112',
  '14:100828092',
  '1:152806897',
  '5:177426834',
  '2:32064013',
  '2:24046995',
  '2:27135283',
  '16:72093343',
  '16:69565534',
  '22:36481511',
  '4:56435793',
  '20:56393569',
  '1:150720623',
  '3:23944598',
  '8:49912188',
  '19:35744207',
  '20:19212030',
  '8:91041204',
  '16:68022870',
  '22:46536744',
  '13:75619945',
  '13:19633629',
  '18:36128956',
  '19:6462987',
  '5:77030475',
  '12:26938971',
  '19:676738',
  '16:2205948',
  '7:74532570',
  '15:45430720',
  '16:89575314',
  '8:94208354',
  '10:49524250',
  '3:49020609',
  '6:32838389',
  '11:74008826',
  '11:33286722',
  '4:26490755',
  '8:94209791',
  '3:10992592',
  '5:140042939',
  '1:100132821',
  '10:116670146',
  '11:43680726',
  '21:41168481',
  '11:57761365',
  '16:81077468',
  '2:207711554',


### 4.3.2 Inspect Dataset Files (.npy and .mmap)

Let's examine the contents of the `.npy` and `.mmap` files in the dataset folder.

In [62]:
# Inspect .npy and .mmap files in the dataset folder
import numpy as np
from pathlib import Path

# Get the dataset folder name (generated from raw data path)
data_path = list(dataset_metrics.keys())[0]
dataset_name = str(Path(data_path).with_suffix("")).replace("/", "_").replace("\\", "_")
dataset_dir = Path(PROCESSED_DIR) / dataset_name

print(f"Dataset folder: {dataset_dir}")
print(f"Files in folder: {list(dataset_dir.glob('*'))}")

Dataset folder: ../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered
Files in folder: [PosixPath('../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered/var.mmap'), PosixPath('../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered/species.npy'), PosixPath('../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered/var_names.npy'), PosixPath('../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered/obs_names.npy'), PosixPath('../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered/X_shape.npy'), PosixPath('../data/tutorials/processed/quick_setup/.._data_tutorials_raw_toy_filtered/X.mmap')]


In [63]:
# 1. Load X_shape.npy - Shape of the methylation array
X_shape = np.load(dataset_dir / "X_shape.npy")
print(f"Shape: {X_shape}")
print(f"Meaning: ({X_shape[0]} samples, {X_shape[1]} CpG sites)")

Shape: [   38 19948]
Meaning: (38 samples, 19948 CpG sites)


In [64]:
X_shape

array([   38, 19948])

In [34]:
# 2. Load species.npy - Species name
species = np.load(dataset_dir / "species.npy", allow_pickle=True)
print(f"Species: {species}")

Species: homo_sapiens


In [38]:
species

array('homo_sapiens', dtype='<U12')

In [36]:
# 3. Load var_names.npy - Column names for var array
var_names = np.load(dataset_dir / "var_names.npy", allow_pickle=True)
print(f"Variable names: {var_names}")

Variable names: ['chrom' 'position']


In [37]:
var_names

array(['chrom', 'position'], dtype='<U8')

In [41]:
# 4. Load obs_names.npy - Sample IDs
obs_names = np.load(dataset_dir / "obs_names.npy", allow_pickle=True)
print(f"Number of samples: {len(obs_names)}")
print(f"Sample IDs: {obs_names}")

Number of samples: 38
Sample IDs: ['GSM5525203' 'GSM5525204' 'GSM5525205' 'GSM5525206' 'GSM5525207'
 'GSM5525208' 'GSM5525209' 'GSM5525210' 'GSM5525211' 'GSM5525212'
 'GSM5525213' 'GSM5525214' 'GSM5525215' 'GSM5525216' 'GSM5525217'
 'GSM5525218' 'GSM5525219' 'GSM5525220' 'GSM5525221' 'GSM5525222'
 'GSM5525223' 'GSM5525224' 'GSM5525225' 'GSM5525226' 'GSM5525227'
 'GSM5525228' 'GSM5525229' 'GSM5525230' 'GSM5525231' 'GSM5525232'
 'GSM5525233' 'GSM5525234' 'GSM5525235' 'GSM5525236' 'GSM5525237'
 'GSM5525238' 'GSM5525239' 'GSM5525240']


In [49]:
obs_names

array(['GSM5525203', 'GSM5525204', 'GSM5525205', 'GSM5525206',
       'GSM5525207', 'GSM5525208', 'GSM5525209', 'GSM5525210',
       'GSM5525211', 'GSM5525212', 'GSM5525213', 'GSM5525214',
       'GSM5525215', 'GSM5525216', 'GSM5525217', 'GSM5525218',
       'GSM5525219', 'GSM5525220', 'GSM5525221', 'GSM5525222',
       'GSM5525223', 'GSM5525224', 'GSM5525225', 'GSM5525226',
       'GSM5525227', 'GSM5525228', 'GSM5525229', 'GSM5525230',
       'GSM5525231', 'GSM5525232', 'GSM5525233', 'GSM5525234',
       'GSM5525235', 'GSM5525236', 'GSM5525237', 'GSM5525238',
       'GSM5525239', 'GSM5525240'], dtype=object)

In [43]:
# 5. Memory-map X.mmap - Methylation values
n_samples, n_features = X_shape
X = np.memmap(dataset_dir / "X.mmap", dtype="float32", mode="r", shape=(n_samples, n_features))

print(f"Shape: {X.shape}")
print(f"Dtype: {X.dtype}")
print(f"Memory-mapped: Data stays on disk until accessed")
print(f"\nFirst sample, first 10 CpG sites:")
print(f"  {X[0, :10]}")
print(f"\nValue range: [{np.nanmin(X):.4f}, {np.nanmax(X):.4f}]")
print(f"Mean (excluding NaN): {np.nanmean(X):.4f}")
print(f"NaN count: {np.isnan(X).sum()} ({np.isnan(X).sum() / X.size * 100:.2f}%)")

Shape: (38, 19948)
Dtype: float32
Memory-mapped: Data stays on disk until accessed

First sample, first 10 CpG sites:
  [0.79132104 0.9053774  0.09116434 0.09065057 0.9517739  0.04833379
 0.93864024 0.03551678 0.05687732 0.08756987]

Value range: [0.0022, 0.9978]
Mean (excluding NaN): 0.2858
NaN count: 38689 (5.10%)


In [48]:
X

memmap([[0.79132104, 0.9053774 , 0.09116434, ..., 0.07747962, 0.04326987,
         0.05758965],
        [0.82889205, 0.96462005, 0.04256871, ..., 0.06956913, 0.03936126,
         0.07051476],
        [       nan,        nan,        nan, ...,        nan,        nan,
                nan],
        ...,
        [0.7495498 , 0.69783205, 0.05310556, ..., 0.09485937, 0.03552543,
         0.07790511],
        [0.7923758 , 0.82061225, 0.04844879, ..., 0.10573225, 0.04325655,
         0.08116442],
        [0.573936  , 0.48310536, 0.09280476, ..., 0.10531504, 0.04024497,
         0.08525321]], shape=(38, 19948), dtype=float32)

In [73]:
# 6. Memory-map var.mmap - Genomic coordinates (chromosome idx, position)
var = np.memmap(dataset_dir / "var.mmap", dtype="int32", mode="r", shape=(n_features, 2))

print(f"Shape: {var.shape}")
print(f"Dtype: {var.dtype}")
print(f"Columns: {var_names} (chromosome index, genomic position)")
print(f"\nFirst 10 CpG sites [chrom_idx, position]:")
for i in range(10):
    print(f"  CpG {i}: chromosome={var[i, 0]}, position={var[i, 1]}")
print(f"\nUnique chromosomes: {np.unique(var[:, 0])}")

Shape: (19948, 2)
Dtype: int32
Columns: ['chrom' 'position'] (chromosome index, genomic position)

First 10 CpG sites [chrom_idx, position]:
  CpG 0: chromosome=7, position=28878778
  CpG 1: chromosome=15, position=57757815
  CpG 2: chromosome=19, position=15686236
  CpG 3: chromosome=11, position=176164344
  CpG 4: chromosome=5, position=93347430
  CpG 5: chromosome=5, position=88824576
  CpG 6: chromosome=7, position=8796567
  CpG 7: chromosome=5, position=20682864
  CpG 8: chromosome=0, position=227734810
  CpG 9: chromosome=18, position=8064259

Unique chromosomes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21]


In [46]:
var

memmap([[        7,  28878778],
        [       15,  57757815],
        [       19,  15686236],
        ...,
        [       15, 128650909],
        [        9,  74292359],
        [       19, 101154313]], shape=(19948, 2), dtype=int32)

### 4.4 Declare data module

Let's define two data modules: one for the forward pass and reconstructing the methylation, and another one the attention weights.

In [23]:
# Define datamodule
quick_setup_datamodule = CpGPTDataModule(
    predict_dir=PROCESSED_DIR,
    dependencies_dir=LLM_DEPENDENCIES_DIR,
    batch_size=1,
    num_workers=0,
    max_length=MAX_INPUT_LENGTH,
    dna_llm=config.data.dna_llm,
    dna_context_len=config.data.dna_context_len,
    sorting_strategy=config.data.sorting_strategy,
    pin_memory=False,
)

# Define datamodule
quick_setup_datamodule_attn = CpGPTDataModule(
    predict_dir=PROCESSED_DIR,
    dependencies_dir=LLM_DEPENDENCIES_DIR,
    batch_size=1,
    num_workers=0,
    max_length=MAX_ATTN_LENGTH,
    dna_llm=config.data.dna_llm,
    dna_context_len=config.data.dna_context_len,
    sorting_strategy=config.data.sorting_strategy,
    pin_memory=False,
)

cpgpt -DNALLMEmbedder: Ensembl metadata dictionary loaded successfully


## 5. Run Inference

There are several ways to perform inference with CpGPT. Here, we'll go through the most common ones. 

Different CUDA versions and GPU architectures may yield slightly different numerical results due to hardware-specific optimizations.

### 5.1 Declare Trainer

Given all models were trained under mixed precision, we'll use the `precision="16-mixed"` argument. However, if you finetune it using a different precision, you can change that accordingly.

In [24]:
trainer = CpGPTTrainer(precision="16-mixed")

HPU available: False, using: 0 HPUs


### 5.2 Get Sample Embeddings

In [25]:
quick_setup_sample_embeddings = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="forward",
    return_keys=["sample_embedding"]
)

RuntimeError: Invalid buffer size: 11.13 GB

In [19]:
quick_setup_sample_embeddings

{'sample_embedding': tensor([[-0.0580, -0.0656, -0.0079,  ..., -0.0076, -0.1528, -0.0810],
         [-0.0750, -0.0756, -0.0059,  ..., -0.0241, -0.1455, -0.0859],
         [ 0.0325, -0.0363, -0.1413,  ..., -0.0782, -0.1645, -0.1452],
         ...,
         [-0.0838, -0.0999, -0.0362,  ...,  0.0130, -0.0523, -0.1371],
         [-0.0597, -0.0813, -0.0225,  ...,  0.0113, -0.1043, -0.0967],
         [-0.0248, -0.0809, -0.0548,  ...,  0.0250, -0.0796, -0.0924]])}

### 5.3 Predict Phenotypes

In [20]:
quick_setup_pred_conditions = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="forward",
    return_keys=["pred_conditions"]
)

cpgpt: CpGPTDataset: Initializing class CpGPTDataset.
cpgpt: CpGPTDataset: Loaded existing dataset metrics.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

In [21]:
quick_setup_pred_conditions

{'pred_conditions': tensor([[ 0.3667],
         [ 0.0154],
         [ 6.9766],
         [ 0.4402],
         [-0.6748],
         [-0.2629],
         [-0.7241],
         [-2.7090],
         [-0.1346],
         [-0.4250],
         [-0.0770],
         [ 0.1849],
         [-0.0803],
         [ 0.4863],
         [ 2.5410],
         [-1.4512],
         [-3.1914],
         [ 3.1328],
         [-0.7524],
         [-1.4854],
         [-1.8594],
         [-1.4404],
         [-2.0391],
         [-1.4297],
         [-2.4863],
         [-2.0703],
         [-2.4922],
         [-2.4121],
         [-2.3438],
         [-1.7637],
         [-1.4941],
         [-2.4941],
         [-0.4998],
         [-0.5352],
         [-1.9775],
         [-3.3359],
         [-1.0107],
         [-0.5669]], dtype=torch.float16)}

### 5.4 Reconstruct Methylation

As an example, let's get some the reconstructed methylation values for some locations of interest based on the Illumina probes.

In [22]:
# Random probes for demonstration
probes = list(df.columns[0:100])

probes[0:5]

['cg00000292', 'cg00002426', 'cg00003994', 'cg00005847', 'cg00008493']

In [23]:
# Convert probes to genomic locations
genomic_locations = prober.locate_probes(probes, "homo_sapiens")

genomic_locations[0:5]

['16:28878778', '3:57757815', '7:15686236', '2:176164344', '14:93347430']

In [24]:
quick_setup_pred_meth = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="reconstruct",
    genomic_locations=genomic_locations,
    species="homo_sapiens",
    return_keys=["pred_meth"],
)

cpgpt: CpGPTDataset: Initializing class CpGPTDataset.
cpgpt: CpGPTDataset: Loaded existing dataset metrics.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Be mindful as the reconstructed values are M values, not beta values. Therefore, you need to convert them to beta values using the `m_to_beta` function.

In [25]:
quick_setup_pred_meth["pred_meth"] = m_to_beta(quick_setup_pred_meth["pred_meth"])
quick_setup_pred_meth

{'pred_meth': tensor([[0.8501, 0.8633, 0.0503,  ..., 0.0348, 0.9292, 0.7095],
         [0.8706, 0.8833, 0.0492,  ..., 0.0340, 0.9419, 0.7275],
         [0.3799, 0.4067, 0.2776,  ..., 0.3308, 0.4133, 0.2937],
         ...,
         [0.7925, 0.7881, 0.0529,  ..., 0.0367, 0.9351, 0.7075],
         [0.8247, 0.8291, 0.0523,  ..., 0.0337, 0.9351, 0.7031],
         [0.6494, 0.4927, 0.0576,  ..., 0.0337, 0.9385, 0.7085]],
        dtype=torch.float16)}

A more powerful way of reconstructing the methylation values is using chain-of-thought. With additional test-time compute, we can let the model "think harder" about the problem, which can lead to better performance. However, it also takes considerably longer dependending on the number of thinking steps.

In [26]:
quick_setup_pred_meth_cot = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule,
    predict_mode="reconstruct",
    genomic_locations=genomic_locations,
    species="homo_sapiens",
    n_thinking_steps=5,
    thinking_step_size=1000,
    uncertainty_quantile=0.1,
    return_keys=["pred_meth"],
)

cpgpt: CpGPTDataset: Initializing class CpGPTDataset.
cpgpt: CpGPTDataset: Loaded existing dataset metrics.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

In [27]:
quick_setup_pred_meth_cot["pred_meth"] = m_to_beta(quick_setup_pred_meth_cot["pred_meth"])
quick_setup_pred_meth_cot

{'pred_meth': tensor([[0.8516, 0.8486, 0.0482,  ..., 0.0344, 0.9248, 0.7080],
         [0.8691, 0.8696, 0.0484,  ..., 0.0330, 0.9434, 0.7383],
         [0.4485, 0.4104, 0.3074,  ..., 0.3823, 0.3540, 0.2976],
         ...,
         [0.7856, 0.7690, 0.0512,  ..., 0.0355, 0.9326, 0.6919],
         [0.8271, 0.8198, 0.0520,  ..., 0.0332, 0.9331, 0.6934],
         [0.6523, 0.4868, 0.0560,  ..., 0.0331, 0.9297, 0.7061]],
        dtype=torch.float16)}

### 5.5 Analyze Attention Weights

The amount of memory required to store the attention weights is enormous. Therefore, we only use 1000 features for the demonstration. Also, remember that the the first token is the CLS token.

In [28]:
quick_setup_attn = trainer.predict(
    model=model,
    datamodule=quick_setup_datamodule_attn,
    predict_mode="attention",
    aggregate_heads="mean",
    layer_index=-1,
    return_keys=["attention_weights", "chroms", "positions", "mask_na", "meth"],
)

cpgpt: CpGPTDataset: Initializing class CpGPTDataset.
cpgpt: CpGPTDataset: Loaded existing dataset metrics.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

In [29]:
quick_setup_attn

{'attention_weights': tensor([[[0.0011, 0.0010, 0.0009,  ...,    nan,    nan,    nan],
          [0.0010, 0.0010, 0.0010,  ...,    nan,    nan,    nan],
          [0.0010, 0.0010, 0.0010,  ...,    nan,    nan,    nan],
          ...,
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan]],
 
         [[0.0011, 0.0010, 0.0010,  ...,    nan,    nan,    nan],
          [0.0010, 0.0011, 0.0010,  ...,    nan,    nan,    nan],
          [0.0010, 0.0010, 0.0010,  ...,    nan,    nan,    nan],
          ...,
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan],
          [   nan,    nan,    nan,  ...,    nan,    nan,    nan]],
 
         [[0.0121, 0.0115, 0.0097,  ...,    nan,    nan,    nan],
          [0.0114, 0.0123, 0.0098,  ...,    nan,    nan,    nan],
          [0.0105, 

: 